In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import skimage.morphology as mo
from skimage import io, color #Scikit-Image
from PIL import Image # Pillow
import cv2
import os
import random
import torch # Will work on using PyTorch here later
from torch.utils.data  import Dataset, DataLoader
from torchvision import transforms
import torchvision.transforms.functional as TF
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import matplotlib.pyplot as plt
import pandas as pd

In [10]:
validation_set_size = 0.2 # Set up the % of data to be validation dataset

# The dataloader basically contains batches of images and their corresponding masks
class Muscle(Dataset):
  def __init__(self, train = True, transformX = None, transformY = None):
    # I have previously created a file named 500_train.csv using the file names. Here we will read in the csv to access data in google drive.
    # hayo: should this be 300_train.csv??
    self.pixel_file = pd.read_csv('/Users/taliacho/Downloads/Ranger Lab/Bryan-Ranger/local_data/300_train.csv')
    self.transformX = transformX
    self.transformY = transformY
    self.train = train

    # Split the dataset to train and validation using sklearn function train_test_split
    self.train_data, self.validation_data = train_test_split(self.pixel_file,
                                                                test_size = validation_set_size,
                                                                random_state = 5)
  def __len__(self):
    if self.train:
      return len(self.train_data)
    return len(self.validation_data)

  def __getitem__(self, index):
    train_path = '/Users/taliacho/Downloads/Ranger Lab/Bryan-Ranger/local_data/train_data'

    if self.train:
      imx_name = os.path.join(train_path, self.train_data.iloc[index, 1])
      imy_name = os.path.join(train_path, self.train_data.iloc[index, 1].replace('.jpeg','_mask.jpg'))
    else:
      imx_name = os.path.join(train_path, self.validation_data.iloc[index, 1])
      imy_name = os.path.join(train_path, self.validation_data.iloc[index, 1].replace('.jpeg','_mask.jpg'))

    # original image
    imx = Image.open(imx_name)

    # mask for the image
    imy = Image.open(imy_name).convert('L')

    if self.train:
      # Random horizontal flipping
      if random.random() > 0.5:
        imx = TF.hflip(imx)
        imy = TF.hflip(imy)

      # Random vertical flipping
      if random.random() > 0.5:
        imx = TF.vflip(imx)
        imy = TF.vflip(imy)

      # Random rotation
      if random.random() > 0.8:
        angle = random.choice([-30, -90, -60, -45 -15, 0, 15, 30, 45, 60, 90])
        imx = TF.rotate(imx, angle)
        imy = TF.rotate(imy, angle)

    # We will use resize, tensorlize, and normalize in the following cell
    if self.transformX :
      imx = self.transformX(imx)
      imy = self.transformY(imy)

    sample = {'image': imx, 'mask': imy}
    return sample

In [11]:
# Init transform functions
tx_X = transforms.Compose([transforms.Resize((256, 256)),
                           transforms.ToTensor(),
                           transforms.Normalize((0.5,), (0.5,))])
tx_Y = transforms.Compose([transforms.Resize((256, 256)),
                           transforms.ToTensor(),  ################ no need to normalize the mask
                           # transforms.Normalize((0.5,), (0.5,))
                          ])
train_data = Muscle(train = True, transformX = tx_X, transformY = tx_Y)
validation_data = Muscle(train = False, transformX = tx_X, transformY = tx_Y )

In [12]:
# Dataloaders
batch_size=8
train_loader = DataLoader(dataset = train_data, batch_size = 8, shuffle = True, num_workers = 2)
validation_loader = DataLoader(dataset = validation_data, batch_size = 8, shuffle = True, num_workers = 2)
print(len(train_loader)) #len(train_loader)*batch_size = total number of images in training set (50*8 = 400), hayo: now 30?
print(len(validation_loader)) #len(validation_loader)*batch_size = total number of images in validation set(13*8 = 104 ~ 100)

#hayo the validation_loader is now 8 instead of 13
print(len(validation_loader)*batch_size)

30
8
64


In [13]:
# The following functions will return numpy array from the transformed tensors which were
# obtained from our train_loader. Plot them and see if they are intact
def im_converterX(tensor):
  image = tensor.cpu().clone().detach().numpy() # make copy of tensor and converting it to numpy
                                              # as we will need original later
  image = image.transpose(1,2,0) # swapping axes making (1, 28, 28) image to a (28, 28, 1)
  print("image shape is ",image.shape)
  image = image * np.array((0.5, 0.5, 0.5)) + np.array((0.5, 0.5, 0.5)) # unnormalizing the image
                                              # this also outputs (28, 28, 3) which seems important for plt.imshow
  image = image.clip(0, 1) # to make sure final values are in range 0 to 1 as .ToTensor outputed
  return image

def im_converterY(tensor):
  image = tensor.cpu().clone().detach().numpy()
  image = image.transpose(1,2,0)
  print("image shape is ",image.shape)
  image = image * np.array((1, 1, 1))
  image = image.clip(0, 1)
  return image

In [14]:
## Here we loop through our train_loader and see the images
fig = plt.figure(figsize = (15,6))

for ith_batch, sample_batched in enumerate(train_loader):
    print(ith_batch, sample_batched['image'].size(), sample_batched['mask'].size())

    for index in range(2):
        ax = fig.add_subplot(2, 2 , index + 1)  # subplot index starts from 1
        plt.imshow(im_converterX(sample_batched['image'][index]))
        ax = fig.add_subplot(2, 2, index + 3)
        plt.imshow(im_converterY(sample_batched['mask'][index]))
    break

Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/Users/taliacho/anaconda3/lib/python3.11/multiprocessing/spawn.py", line 122, in spawn_main
Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/Users/taliacho/anaconda3/lib/python3.11/multiprocessing/spawn.py", line 122, in spawn_main
    exitcode = _main(fd, parent_sentinel)
    exitcode = _main(fd, parent_sentinel)
                          ^ ^ ^ ^ ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/taliacho/anaconda3/lib/python3.11/multiprocessing/spawn.py", line 132, in _main
^^^^^^^
  File "/Users/taliacho/anaconda3/lib/python3.11/multiprocessing/spawn.py", line 132, in _main
    self = reduction.pickle.load(from_parent)
    self = reduction.pickle.load(from_parent)
                 ^ ^ ^ ^ ^ ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
^^^^^
AttributeError: Can't get attribute 'Muscle' on <module '__main__' (built-in)>
AttributeError: Can't get attri

RuntimeError: DataLoader worker (pid(s) 58624, 58625) exited unexpectedly

<Figure size 1500x600 with 0 Axes>

In [15]:
class double_conv(nn.Module):
  '''(conv => BN => ReLU) * 2'''
  def __init__(self, in_ch, out_ch):
    super(double_conv, self).__init__()
    self.conv = nn.Sequential(
      nn.Conv2d(in_ch, out_ch, 3, padding=1),
      nn.BatchNorm2d(out_ch),
      nn.ReLU(inplace=True),
      nn.Conv2d(out_ch, out_ch, 3, padding=1),
      nn.BatchNorm2d(out_ch),
      nn.ReLU(inplace=True)
    )

  def forward(self, x):
    x = self.conv(x)
    return x

class inconv(nn.Module):
  def __init__(self, in_ch, out_ch):
    super(inconv, self).__init__()
    self.conv = double_conv(in_ch, out_ch)

  def forward(self, x):
    x = self.conv(x)
    return x

class down(nn.Module):
  def __init__(self, in_ch, out_ch):
    super(down, self).__init__()
    self.mpconv = nn.Sequential(
        nn.MaxPool2d(2),
        double_conv(in_ch, out_ch)
    )

  def forward(self, x):
    x = self.mpconv(x)
    return x


class up(nn.Module):
  def __init__(self, in_ch, out_ch, bilinear=True):
    super(up, self).__init__()
    self.up = nn.Upsample(
      scale_factor=2, mode='bilinear', align_corners=True)
    self.up = nn.ConvTranspose2d(in_ch // 2, in_ch // 2, kernel_size=2, stride=2)
    self.conv = double_conv(in_ch, out_ch)

  def forward(self, x1, x2):
    x1 = self.up(x1)
    diff1 = x2.shape[2]-x1.shape[2]
    diff2 = x2.shape[3]-x1.shape[3]
    x1 = F.pad(x1, pad=(diff1//2, diff1-diff1//2, diff2//2, diff2-diff2//2))
    x = torch.cat([x2, x1], dim=1)
    x = self.conv(x)
    return x


class outconv(nn.Module):
  def __init__(self, in_ch, out_ch):
    super(outconv, self).__init__()
    self.conv = nn.Conv2d(in_ch, out_ch, 1)

  def forward(self, x):
    x = self.conv(x)
    return x

In [16]:
class UNet(nn.Module):
  def __init__(self, n_channels, n_classes):
    super(UNet, self).__init__()
    self.inc = inconv(n_channels, 64)
    self.down1 = down(64, 128)
    self.down2 = down(128, 256)
    self.down3 = down(256, 512)
    self.down4 = down(512, 512)
    self.up1 = up(1024, 256, bilinear = False)
    self.up2 = up(512, 128, bilinear = False)
    self.up3 = up(256, 64, bilinear = False)
    self.up4 = up(128, 64, bilinear = False)
    self.outc = outconv(64, n_classes)
    self.dropout = torch.nn.Dropout2d(0.5)

  def forward(self, x):
    x = x.float()
    x1 = self.inc(x)
    x2 = self.down1(x1)
    x3 = self.down2(x2)
    x4 = self.down3(x3)
    x5 = self.down4(x4)
    x = self.up1(x5, x4)
    x = self.up2(x, x3)
    x = self.dropout(x)
    x = self.up3(x, x2)
    x = self.up4(x, x1)
    x = self.outc(x)
    return torch.sigmoid(x)

In [17]:
model = UNet(3, 1)
# hayo: had to edit model.to('cuda') bc Apple hasn't supported NVIDIA graphics cards natively in macOS since macOS Mojave
model.to('cpu')
print("Model Loaded to GPU")

Model Loaded to GPU


In [18]:
criterion = nn.BCELoss() # BCE = Binary Cross Entropy
optimizer = torch.optim.Adam(model.parameters(), lr = 0.01) # lr = learning rate

In [19]:
# calculates similarity index between predicted and actual segmentation
def dice_index(y_pred, y_actual):
  smooth = 0.000001 # prevent division by 0
  size_of_batch = y_pred.size(0)

  p1 = y_pred.view(size_of_batch, -1)
  p2 = y_actual.view(size_of_batch, -1)

  intersection = (p1 * p2).sum()

  dice =  ((2.0 * intersection )+ smooth) / (p1.sum() + p2.sum() + smooth)
  #dice.requires_grad = True

  return dice

# calculate dice loss which will be later used in loss function calculation
def dice_loss(y_predict, y_train): ## to add in bce looss
  return 1 -(dice_index(y_predict, y_train))

In [20]:
epochs = 40 # number of iterations of the training dataset during the training process

train_running_loss_history = []
validation_running_loss_history = []

model.to('cuda')

for e in range(epochs):
  train_running_loss = 0.0
  validation_running_loss = 0.0
  model.train()
  for ith_batch, sample_batched in enumerate(train_loader):
    X_train = sample_batched['image'].to('cuda')
    y_train = sample_batched['mask'].to('cuda') # hayo: make the weight to cuda
    optimizer.zero_grad()
    y_pred = model(X_train) # hayo: Input type (torch.cuda.FloatTensor) and weight type (torch.FloatTensor) should be the same
    loss = 0.30 * dice_loss(y_pred, y_train) +  0.70 * criterion(y_pred, y_train)
    loss.backward()
    optimizer.step()
    if ith_batch % 5 == 0:
      print('Epoch: ', e + 1, 'Batch: ', ith_batch, 'Current Loss: ', loss.item())
    train_running_loss += loss.item()
  else:
    with torch.no_grad():
      model.eval()
      for ith_batch, sample_batched in enumerate(validation_loader):
        X_val = sample_batched['image'].to('cuda')
        y_val = sample_batched['mask'].to('cuda')
        y_out = model(X_val)
        out_val = (y_out + 0.5).int().float()
        val_loss = 0.3 * dice_loss(out_val, y_val)  + 0.7 * criterion(y_out, y_val)
        validation_running_loss += val_loss.item()
      print("================================================================================")
      print("Epoch {} completed".format(e + 1))

      train_epoch_loss = train_running_loss / len(train_loader)
      validation_epoch_loss = validation_running_loss / len(validation_loader)

      print("Average train loss is {}: ".format(train_epoch_loss))
      print("Average validation loss is {}".format(validation_epoch_loss))
      print("================================================================================")
      train_running_loss_history.append(train_epoch_loss)
      validation_running_loss_history.append(validation_epoch_loss)

  torch.cuda.empty_cache()

AssertionError: Torch not compiled with CUDA enabled